In [24]:
# imports
from transformers import AutoProcessor, AutoModelForCausalLM
import torch
import os
from tqdm import tqdm
import shutil
import random
import pandas as pd
from glob import glob
import numpy as np
import seaborn as sns
from PIL import Image, ImageFilter, ImageEnhance
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.image as mpimg
import cv2
from ultralytics import YOLO
import clip
import re
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# ML / evaluation (fusion classifier)
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Utils
from collections import defaultdict
from typing import List, Dict, Tuple, Optional

In [2]:
# Use GPU if available
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

True
Using device: cuda


In [27]:
clip_model, preprocess = clip.load("ViT-B/32", device=device)

In [4]:
yolo_model = YOLO("yolov8m.pt")

In [8]:
UCF_ROOT = Path(r"UCF Crime Dataset")
SPLIT = "Train"
NORMAL_CLASS = "NormalVideos"

split_dir = UCF_ROOT / SPLIT
print("UCF_ROOT exists:", UCF_ROOT.exists())
print("split_dir exists:", split_dir.exists())
print("classes:", [p.name for p in split_dir.iterdir() if p.is_dir()])

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

UCF_ROOT exists: True
split_dir exists: True
classes: ['Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'NormalVideos', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']


In [10]:
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
_LAST_NUM = re.compile(r"_(\d+)$")  # Abuse001_x264_120 -> 120

def frame_index(p: Path) -> int:
    m = _LAST_NUM.search(p.stem)
    return int(m.group(1)) if m else 0

In [11]:
def list_videos(split_dir: Path):
    items = []
    class_dirs = sorted([d for d in split_dir.iterdir() if d.is_dir()])

    for class_dir in class_dirs:
        video_dirs = sorted([v for v in class_dir.iterdir() if v.is_dir()])
        for video_dir in video_dirs:
            frames = [p for p in video_dir.iterdir()
                      if p.is_file() and p.suffix.lower() in IMG_EXTS]
            if not frames:
                continue
            frames = sorted(frames, key=frame_index)

            items.append({
                "class_name": class_dir.name,
                "video_id": video_dir.name,
                "video_dir": video_dir,
                "frames": frames
            })
    return items

videos = list_videos(split_dir)
print("Total videos:", len(videos))
print("Example item:", videos[0]["class_name"], videos[0]["video_id"], "frames:", len(videos[0]["frames"]))


Total videos: 1610
Example item: Abuse Abuse001_x264 frames: 273


In [15]:
classes = sorted({v["class_name"] for v in videos})
label2idx = {c:i for i,c in enumerate(classes)}
idx2label = {i:c for c,i in label2idx.items()}

print("Classes:", classes)
print("label2idx:", label2idx)


Classes: ['Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'NormalVideos', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']
label2idx: {'Abuse': 0, 'Arrest': 1, 'Arson': 2, 'Assault': 3, 'Burglary': 4, 'Explosion': 5, 'Fighting': 6, 'NormalVideos': 7, 'RoadAccidents': 8, 'Robbery': 9, 'Shooting': 10, 'Shoplifting': 11, 'Stealing': 12, 'Vandalism': 13}


In [17]:
def uniform_sample_indices(n_frames: int, n_sample: int) -> np.ndarray:
    if n_frames <= 0:
        return np.array([], dtype=int)
    if n_frames >= n_sample:
        return np.linspace(0, n_frames - 1, num=n_sample, dtype=int)
    return np.linspace(0, n_frames - 1, num=n_sample, dtype=int) 


In [21]:
class UCFCrimeVideoDataset(Dataset):
    def __init__(self, video_items, label2idx, n_frames=32, transform=None,
                 return_binary=False, normal_class="NormalVideos"):
        self.video_items = video_items
        self.label2idx = label2idx
        self.n_frames = n_frames
        self.transform = transform
        self.return_binary = return_binary
        self.normal_class = normal_class

    def __len__(self):
        return len(self.video_items)

    def __getitem__(self, i):
        item = self.video_items[i]
        frames = item["frames"]
        idxs = uniform_sample_indices(len(frames), self.n_frames)

        imgs = []
        for k in idxs:
            img = Image.open(frames[int(k)]).convert("RGB")
            if self.transform is not None:
                img = self.transform(img)  # tensor
            imgs.append(img)

        if self.transform is not None:
            x = torch.stack(imgs, dim=0)  # [T,3,H,W]
        else:
            x = imgs  # list of PIL

        class_name = item["class_name"]
        y_multi = self.label2idx[class_name]
        y_bin = 0 if class_name == self.normal_class else 1

        if self.return_binary:
            return x, y_bin, y_multi, item["video_id"], class_name
        return x, y_multi, item["video_id"], class_name


In [ ]:
from torch.utils.data import DataLoader

ds = UCFCrimeVideoDataset(
    videos, label2idx,
    n_frames=32,
    transform=preprocess,        
    return_binary=True,
    normal_class=NORMAL_CLASS
)

dl = DataLoader(ds, batch_size=2, shuffle=True, num_workers=0)

x, y_bin, y_multi, video_ids, class_names = next(iter(dl))

print("x shape:", x.shape)  # expected: [B, T, 3, 224, 224]
print("binary:", y_bin)
print("multi:", y_multi)
print("video_ids:", video_ids)
print("class_names:", class_names)


x shape: torch.Size([2, 32, 3, 224, 224])
binary: tensor([0, 1])
multi: tensor([7, 9])
video_ids: ('Normal_Videos649_x264', 'Robbery105_x264')
class_names: ('NormalVideos', 'Robbery')
